# Day 56 Project — Solution: Doc-Upload AI App

Upload a file → extract text → ask the AI about it.

**Deliverable:** `doc_api.py` — run with `uvicorn doc_api:app --reload`.

In [ ]:
# ── provided source string ──────────────────────────────────────────────────
_DOC_API_SRC = '"""doc_api.py — Day 056 project: FastAPI with file upload + AI Q&A.\n\nRun:  uvicorn doc_api:app --reload\nDocs: http://localhost:8000/docs\n"""\nimport io\nimport re\nimport secrets\nfrom pathlib import Path\nfrom typing import Annotated\n\nimport pypdf\nfrom fastapi import FastAPI, File, HTTPException, UploadFile\nfrom fastapi.middleware.cors import CORSMiddleware\nimport ollama\n\n# --- config ---------------------------------------------------------------\nUPLOAD_DIR  = Path("uploads")\nALLOWED_TYPES = {"text/plain", "application/pdf"}\nMAX_SIZE    = 5 * 1024 * 1024   # 5 MB\nMAX_DOC_CHARS = 4000             # chars sent to LLM\nMODEL = "llama3.2"\n\n# --- helpers --------------------------------------------------------------\n\ndef validate_upload(content: bytes, filename: str,\n                    allowed_types: set[str], content_type: str,\n                    max_bytes: int) -> tuple[bool, str]:\n    if len(content) == 0:\n        return False, "File is empty"\n    if len(content) > max_bytes:\n        return False, f"File too large ({len(content)} bytes, max {max_bytes})"\n    ext = Path(filename).suffix.lower()\n    if content_type not in allowed_types and ext not in {".txt", ".pdf"}:\n        return False, f"Unsupported type: {content_type}"\n    return True, ""\n\n\ndef safe_filename(original: str) -> str:\n    name = Path(original).name\n    name = re.sub(r"[^\\w\\-.]", "_", name)\n    return f"{secrets.token_hex(4)}_{name}"\n\n\ndef save_upload(content: bytes, filename: str, upload_dir: Path) -> Path:\n    upload_dir.mkdir(parents=True, exist_ok=True)\n    dest = upload_dir / safe_filename(filename)\n    dest.write_bytes(content)\n    return dest\n\n\ndef extract_text(content: bytes, content_type: str) -> str:\n    if content_type == "application/pdf" or content_type.endswith("pdf"):\n        reader = pypdf.PdfReader(io.BytesIO(content))\n        return "\\n".join(p.extract_text() or "" for p in reader.pages)\n    return content.decode("utf-8", errors="replace")\n\n\ndef build_doc_prompt(document_text: str, question: str,\n                     max_doc_chars: int = MAX_DOC_CHARS) -> str:\n    snippet = document_text[:max_doc_chars]\n    return (\n        "You are a helpful assistant. Answer the question based only on the "\n        "document below. If the answer is not in the document, say so.\\n\\n"\n        f"DOCUMENT:\\n{snippet}\\n\\n"\n        f"QUESTION: {question}"\n    )\n\n# --- in-memory store ------------------------------------------------------\n_docs: dict[str, dict] = {}   # doc_id -> {filename, text}\n\n# --- app ------------------------------------------------------------------\napp = FastAPI(title="Doc Upload AI API", version="1.0")\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["http://localhost:8501"],\n    allow_credentials=True,\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\n@app.post("/upload", status_code=201)\nasync def upload_doc(file: UploadFile = File(...)):\n    content = await file.read()\n    ok, err = validate_upload(\n        content, file.filename or "unnamed",\n        ALLOWED_TYPES, file.content_type or "", MAX_SIZE\n    )\n    if not ok:\n        raise HTTPException(status_code=400, detail=err)\n    text   = extract_text(content, file.content_type or "text/plain")\n    doc_id = secrets.token_urlsafe(8)\n    _docs[doc_id] = {"filename": file.filename, "text": text}\n    save_upload(content, file.filename or "unnamed", UPLOAD_DIR)\n    return {"doc_id": doc_id, "filename": file.filename, "chars": len(text)}\n\n@app.get("/documents")\ndef list_documents():\n    return {"documents": [{"doc_id": k, "filename": v["filename"]}\n                          for k, v in _docs.items()]}\n\n@app.post("/ask/{doc_id}")\ndef ask(doc_id: str, question: str):\n    entry = _docs.get(doc_id)\n    if entry is None:\n        raise HTTPException(status_code=404, detail="Document not found")\n    prompt = build_doc_prompt(entry["text"], question)\n    reply  = ollama.chat(model=MODEL,\n                         messages=[{"role": "user", "content": prompt}])["message"]["content"]\n    return {"reply": reply, "doc_id": doc_id}\n\nif __name__ == "__main__":\n    import uvicorn\n    uvicorn.run(app, host="0.0.0.0", port=8000)\n'

# ── write_doc_api ───────────────────────────────────────────────────────────
from pathlib import Path

def write_doc_api(path: str) -> str:
    Path(path).write_text(_DOC_API_SRC, encoding="utf-8")
    return path

out = write_doc_api("doc_api.py")
print(f"Generated: {out}  ({len(_DOC_API_SRC)} chars)")
print(Path(out).read_text(encoding="utf-8")[:120] + "...")


In [ ]:
# ── smoke-test the upload flow with TestClient ──────────────────────────────
import io
import re
import secrets
from pathlib import Path
import pypdf
from fastapi import FastAPI, File, HTTPException, UploadFile
from starlette.testclient import TestClient

# --- replicate doc_api internals in-process (no ollama for checks) -----------

def validate_upload(content, filename, allowed_types, content_type, max_bytes):
    if len(content) == 0:
        return False, "File is empty"
    if len(content) > max_bytes:
        return False, f"File too large"
    ext = Path(filename).suffix.lower()
    if content_type not in allowed_types and ext not in {".txt", ".pdf"}:
        return False, f"Unsupported type: {content_type}"
    return True, ""

def safe_filename(original):
    name = Path(original).name
    name = re.sub(r"[^\w\-.]", "_", name)
    return f"{secrets.token_hex(4)}_{name}"

def extract_text(content, content_type):
    if "pdf" in content_type:
        reader = pypdf.PdfReader(io.BytesIO(content))
        return "\n".join(p.extract_text() or "" for p in reader.pages)
    return content.decode("utf-8", errors="replace")

ALLOWED = {"text/plain", "application/pdf"}
MAX_SIZE = 5 * 1024 * 1024
_docs: dict = {}

app = FastAPI(title="Doc Upload AI API (test)")

@app.post("/upload", status_code=201)
async def upload_doc(file: UploadFile = File(...)):
    content = await file.read()
    ok, err = validate_upload(content, file.filename or "unnamed",
                              ALLOWED, file.content_type or "", MAX_SIZE)
    if not ok:
        raise HTTPException(status_code=400, detail=err)
    text   = extract_text(content, file.content_type or "text/plain")
    doc_id = secrets.token_urlsafe(8)
    _docs[doc_id] = {"filename": file.filename, "text": text}
    return {"doc_id": doc_id, "filename": file.filename, "chars": len(text)}

@app.get("/documents")
def list_documents():
    return {"documents": [{"doc_id": k, "filename": v["filename"]} for k, v in _docs.items()]}

# ── run checks ──────────────────────────────────────────────────────────────
client = TestClient(app, raise_server_exceptions=False)
score = 0; total = 5

def chk(n, ok, msg):
    global score
    print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
    if ok: score += 1

# 1. upload text file → 201
r = client.post("/upload",
                files={"file": ("readme.txt", b"This document talks about clouds.", "text/plain")})
chk(1, r.status_code == 201, f"POST /upload text → 201 (got {r.status_code})")

data = r.json() if r.status_code == 201 else {}
chk(2, isinstance(data.get("doc_id"), str) and len(data.get("doc_id", "")) > 4,
    f"response has doc_id string (got {data.get('doc_id')!r})")

chk(3, data.get("chars", 0) > 0,
    f"chars > 0 (got {data.get('chars')})")

# 4. GET /documents includes our doc
r2 = client.get("/documents")
docs_list = r2.json().get("documents", [])
our_id    = data.get("doc_id", "")
chk(4, any(d["doc_id"] == our_id for d in docs_list),
    f"GET /documents includes uploaded doc_id (found {len(docs_list)} docs)")

# 5. upload invalid type → 400
r3 = client.post("/upload",
                 files={"file": ("photo.jpg", b"\xff\xd8\xff", "image/jpeg")})
chk(5, r3.status_code == 400,
    f"unsupported type → 400 (got {r3.status_code})")

print(f"\nScore: {score} / {total}")
if score == total:
    print("\nDay 56 — File Uploads & Storage complete! 🎉")
print(f"\nDeliverable: doc_api.py generated ({len(_DOC_API_SRC)} chars)")
print("Run:  uvicorn doc_api:app --reload")
print("Docs: http://localhost:8000/docs")
